In [1]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import roc_auc_score, f1_score, classification_report
from xgboost import XGBClassifier
import warnings
warnings.filterwarnings('ignore')

print("Libraries loaded! ✅")

Libraries loaded! ✅


In [2]:
df = pd.read_csv('/kaggle/input/notebooks/sudhamag/preprocessing/preprocessed_data.csv')

X = df.drop('TARGET', axis=1)
y = df['TARGET']

print("Shape:", df.shape)
print("Target distribution:")
print(y.value_counts())

Shape: (307511, 126)
Target distribution:
TARGET
0    282686
1     24825
Name: count, dtype: int64


In [3]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y  # important for imbalanced data!
)

print(f"Train size: {X_train.shape}")
print(f"Test size: {X_test.shape}")

Train size: (246008, 125)
Test size: (61503, 125)


In [4]:
# Calculate scale_pos_weight for XGBoost
neg = (y_train == 0).sum()
pos = (y_train == 1).sum()
scale = neg / pos

print(f"Negative cases: {neg}")
print(f"Positive cases: {pos}")
print(f"Scale pos weight: {scale:.2f}")

Negative cases: 226148
Positive cases: 19860
Scale pos weight: 11.39


In [5]:
print("Training Logistic Regression...")

lr = LogisticRegression(
    max_iter=1000,
    class_weight='balanced',
    random_state=42
)
lr.fit(X_train, y_train)

lr_pred = lr.predict_proba(X_test)[:, 1]
lr_auc = roc_auc_score(y_test, lr_pred)
lr_f1 = f1_score(y_test, lr.predict(X_test))

print(f"Logistic Regression ROC-AUC: {lr_auc:.4f}")
print(f"Logistic Regression F1 Score: {lr_f1:.4f}")

Training Logistic Regression...
Logistic Regression ROC-AUC: 0.7469
Logistic Regression F1 Score: 0.2596


In [6]:
print("Training Random Forest...")

rf = RandomForestClassifier(
    n_estimators=100,
    class_weight='balanced',
    max_depth=10,
    min_samples_leaf=50,
    random_state=42,
    n_jobs=-1
)
rf.fit(X_train, y_train)

rf_pred = rf.predict_proba(X_test)[:, 1]
rf_auc = roc_auc_score(y_test, rf_pred)
rf_f1 = f1_score(y_test, rf.predict(X_test))

print(f"Random Forest ROC-AUC: {rf_auc:.4f}")
print(f"Random Forest F1 Score: {rf_f1:.4f}")

Training Random Forest...
Random Forest ROC-AUC: 0.7371
Random Forest F1 Score: 0.2602


In [7]:
'''print("Training Random Forest...")

rf = RandomForestClassifier(
    n_estimators=100,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)
rf.fit(X_train, y_train)

rf_pred = rf.predict_proba(X_test)[:, 1]
rf_auc = roc_auc_score(y_test, rf_pred)
rf_f1 = f1_score(y_test, rf.predict(X_test))

print(f"Random Forest ROC-AUC: {rf_auc:.4f}")
print(f"Random Forest F1 Score: {rf_f1:.4f}")'''

'print("Training Random Forest...")\n\nrf = RandomForestClassifier(\n    n_estimators=100,\n    class_weight=\'balanced\',\n    random_state=42,\n    n_jobs=-1\n)\nrf.fit(X_train, y_train)\n\nrf_pred = rf.predict_proba(X_test)[:, 1]\nrf_auc = roc_auc_score(y_test, rf_pred)\nrf_f1 = f1_score(y_test, rf.predict(X_test))\n\nprint(f"Random Forest ROC-AUC: {rf_auc:.4f}")\nprint(f"Random Forest F1 Score: {rf_f1:.4f}")'

In [8]:
print("Training XGBoost...")

xgb = XGBClassifier(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.05,
    scale_pos_weight=scale,
    random_state=42,
    eval_metric='auc',
    n_jobs=-1
)
xgb.fit(X_train, y_train)

xgb_pred = xgb.predict_proba(X_test)[:, 1]
xgb_auc = roc_auc_score(y_test, xgb_pred)
xgb_f1 = f1_score(y_test, xgb.predict(X_test))

print(f"XGBoost ROC-AUC: {xgb_auc:.4f}")
print(f"XGBoost F1 Score: {xgb_f1:.4f}")

Training XGBoost...
XGBoost ROC-AUC: 0.7577
XGBoost F1 Score: 0.2726


In [9]:
print("Building Ensemble...")

ensemble_pred = (lr_pred + rf_pred + xgb_pred) / 3
ensemble_auc = roc_auc_score(y_test, ensemble_pred)

print(f"Ensemble ROC-AUC: {ensemble_auc:.4f}")

Building Ensemble...
Ensemble ROC-AUC: 0.7563


In [10]:
results = pd.DataFrame({
    'Model': ['Logistic Regression', 'Random Forest', 'XGBoost', 'Ensemble'],
    'ROC-AUC': [lr_auc, rf_auc, xgb_auc, ensemble_auc],
    'F1 Score': [lr_f1, rf_f1, xgb_f1, '-']
})

print("=== Model Comparison ===")
print(results.to_string(index=False))

=== Model Comparison ===
              Model  ROC-AUC  F1 Score
Logistic Regression 0.746945  0.259645
      Random Forest 0.737116  0.260189
            XGBoost 0.757734  0.272558
           Ensemble 0.756313         -


In [11]:
from scipy.stats import ks_2samp

# KS Statistic for best model (XGBoost)
defaults = xgb_pred[y_test == 1]
non_defaults = xgb_pred[y_test == 0]

ks_stat, ks_pvalue = ks_2samp(defaults, non_defaults)

print(f"KS Statistic: {ks_stat:.4f}")
print(f"KS P-Value: {ks_pvalue:.4f}")
print("\nModel Training Complete! ✅")

KS Statistic: 0.3832
KS P-Value: 0.0000

Model Training Complete! ✅
